In [ ]:
import requests
import pandas as pd

# Full repository tree
url = "https://api.github.com/repos/fivethirtyeight/data/git/trees/master?recursive=1"

response = requests.get(url)
response.raise_for_status()

repo_tree = response.json()

print("GitHub tree truncated:", repo_tree.get("truncated", False))

# All files in the repository
files = [
    item["path"]
    for item in repo_tree["tree"]
    if item["type"] == "blob"
]

# All CSV files
csv_files = [
    path
    for path in files
    if path.lower().endswith(".csv")
]

# index.csv is a catalog, not a dataset
dataset_csv_files = [
    path
    for path in csv_files
    if path != "index.csv"
]

print("All CSV files:", len(csv_files))
print("Dataset CSV files (excluding index.csv):", len(dataset_csv_files))

GitHub tree truncated: False
All CSV files: 505
Dataset CSV files (excluding index.csv): 504


In [ ]:
# Folder containing each CSV file
folders = [
    path.split("/")[0]
    for path in dataset_csv_files
    if "/" in path
]

print("Folders containing CSV files:", len(set(folders)))

Folders containing CSV files: 138


In [ ]:
from collections import Counter

folder_counts = Counter(folders)

multiple_csv = pd.DataFrame(
    [
        {
            "Project folder": folder,
            "CSV files": count
        }
        for folder, count in folder_counts.items()
        if count > 1
    ]
).sort_values(
    "CSV files",
    ascending=False
)

multiple_csv.head(20)

,Project folder,CSV files
47,womens-world-cup-predictions,88
48,world-cup-predictions,84
10,march-madness-predictions,62
28,pollster-ratings,14
27,polls,12
46,us-weather-history,10
15,most-common-name,9
33,puerto-rico-media,8
6,gop-delegate-benchmarks-2024,8
38,state-of-the-polls-2024,8


In [ ]:
!git clone --depth 1 https://github.com/fivethirtyeight/data.git fivethirtyeight_data

Cloning into 'fivethirtyeight_data'...
remote: Enumerating objects: 1086, done.
remote: Counting objects: 100% (1086/1086), done.
remote: Compressing objects: 100% (869/869), done.
remote: Total 1086 (delta 210), reused 760 (delta 196), pack-reused 0 (from 0)
Receiving objects: 100% (1086/1086), 72.40 MiB | 12.50 MiB/s, done.
Resolving deltas: 100% (210/210), done.
Updating files: 100% (899/899), done.


In [ ]:
#CHECK: count rows locally
from pathlib import Path
import csv
import pandas as pd
from tqdm.auto import tqdm

repo_path = Path("fivethirtyeight_data")

csv_files_local = [
    path
    for path in repo_path.rglob("*.csv")
    if str(path.relative_to(repo_path)) != "index.csv"
]

print("CSV files found:", len(csv_files_local))

row_results = []

for path in tqdm(csv_files_local):

    try:
        with open(
            path,
            "r",
            encoding="utf-8-sig",
            errors="replace",
            newline=""
        ) as f:

            reader = csv.reader(f)

            # Skip header
            next(reader, None)

            row_count = sum(1 for _ in reader)

        row_results.append({
            "path": str(path.relative_to(repo_path)),
            "rows": row_count
        })

    except Exception as e:

        row_results.append({
            "path": str(path.relative_to(repo_path)),
            "rows": None,
            "error": str(e)
        })

rows_df = pd.DataFrame(row_results)

print("CSV files checked:", len(rows_df))
print("Successfully counted:", rows_df["rows"].notna().sum())
print("Failed:", rows_df["rows"].isna().sum())

CSV files found: 504


  0%|          | 0/504 [00:00<?, ?it/s]

CSV files checked: 504
Successfully counted: 504
Failed: 0


In [ ]:
valid_rows = rows_df["rows"].dropna()

print(f"Average rows: {valid_rows.mean():,.0f}")
print(f"Median rows: {valid_rows.median():,.0f}")
print(f"Minimum rows: {valid_rows.min():,.0f}")
print(f"Maximum rows: {valid_rows.max():,.0f}")
print(f"Total rows: {valid_rows.sum():,.0f}")

Average rows: 5,970
Median rows: 68
Minimum rows: 2
Maximum rows: 752,351
Total rows: 3,008,832


In [ ]:
largest_by_rows = rows_df.sort_values(
    "rows",
    ascending=False
)

largest_by_rows.head(3)

,path,rows
302,redlining/zone-block-matches.csv,752351
294,polls/pres_primary_avgs_1980-2016.csv,301695
394,twitter-ratio/senators.csv,288615
